# 5. Многоагентная проверка

**Цели:** разделить роли исследователя, скептика и аудитора воспроизводимости. Независимые роли уменьшают риск согласиться с первым правдоподобным ответом, но не гарантируют истину.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/fractal_extreme_series.csv"
SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1
# Graph-EVT-agent is the analysis library; no model code is copied into this repository.
from graph_evt_agent import ResearchAgent, ResearchTask, LLMConfig

llm = LLMConfig.from_env()  # provider/model/key are selected with environment variables
agent = ResearchAgent(llm=llm, seed=SEED)


In [ ]:
from graph_evt_agent import ResearchTeam
team=ResearchTeam(
 agents={
  "analyst": agent.with_role("EVT analyst"),
  "skeptic": agent.with_role("adversarial statistical reviewer"),
  "auditor": agent.with_role("reproducibility auditor"),
 },
 seed=SEED,
)
review=team.run(ResearchTask(
 question="Проведи и раскритикуй EVT-исследование fractal_extreme_series.csv", data=x,
 methods=["pot","declustering","block_bootstrap"],
 constraints={"cite_artifacts":True,"no_uncomputed_claims":True,"seed":SEED},
))
review.consensus


In [ ]:
review.save(ROOT/'results/05_peer_review')
review.disagreements


Особенно ценны не совпадения, а зафиксированные разногласия и предложенные проверки.

**Самопроверка:** чем роль отличается от независимой модели? Как обнаружить неподтверждённое число? Почему консенсус может ошибаться?